# Part 1: Data Understanding

## 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from pandas.plotting import scatter_matrix
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1.2 Load Dataset

In [ ]:
df = pd.read_csv("wine-clustering.csv")
print("Dataset loaded successfully")


## 1.3 Dataset Overview

In [ ]:
print("Dataset Shape:")
print(df.shape)
print("\nFirst 5 Rows:")
display(df.head())
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

## 1.4 Missing Values and Duplicates

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print("\nTotal Missing Values:")
print(df.isnull().sum().sum())
print("\nDuplicate Rows:")
print(df.duplicated().sum())

## 1.5 Summary Statistics

In [ ]:
print("Summary Statistics:")
display(df.describe())
print("\nInterpretation:")
print("The dataset contains chemical measurements of wines.")
print("PCA can reduce correlated chemical features into fewer principal components.")

# Part 2: EDA

## 2.1 Histograms

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    df[col].hist(bins=25)
    plt.title("Histogram of " + col)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()
print("Histograms show the distribution of each feature.")

## 2.2 Boxplots

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    df.boxplot(column=col)
    plt.title("Boxplot of " + col)
    plt.ylabel(col)
    plt.show()
print("Boxplots help identify outliers and feature spread.")

## 2.3 Correlation Heatmap

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(10,8))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.show()
print("The heatmap shows relationships between features.")
print("Highly correlated features create redundancy, which PCA can reduce.")

## 2.4 Pair Plot

In [ ]:
selected_cols = numeric_cols[:5]
scatter_matrix(df[selected_cols], figsize=(10,10), diagonal="hist")
plt.suptitle("Pair Plot of Selected Features")
plt.show()
print("Pair plot shows relationships among selected features.")

## 2.5 EDA Questions

In [ ]:
print("EDA Questions and Answers:")
print("1. Are there missing values? Checked using isnull().sum().")
print("2. Are there duplicates? Checked using duplicated().sum().")
print("3. Are there outliers? Checked using boxplots.")
print("4. Are features correlated? Checked using correlation heatmap.")
print("5. Why PCA? PCA reduces correlated and redundant features.")

# Part 3: Preprocessing

## 3.1 Remove Duplicates

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
print("Original Shape:", df.shape)
print("Shape After Removing Duplicates:", df_clean.shape)

## 3.2 Handle Missing Values

In [ ]:
for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
print("Missing Values After Treatment:")
print(df_clean.isnull().sum().sum())

## 3.3 Standardize Features

In [ ]:
X = df_clean.select_dtypes(include=["int64", "float64"])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print("Scaling Completed")
display(X_scaled_df.head())

## 3.4 Why Scaling is Required

In [ ]:
print("PCA is affected by feature scale.")
print("Features with large values can dominate PCA.")
print("StandardScaler gives every feature mean 0 and standard deviation 1.")
print("This allows PCA to treat all features fairly.")

# Part 4: Correlation Analysis

## 4.1 Correlation Matrix

In [ ]:
correlation_matrix = X.corr()
display(correlation_matrix)

## 4.2 Highly Correlated Features

In [ ]:
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        value = correlation_matrix.iloc[i, j]
        if abs(value) > 0.70:
            high_corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], value))
high_corr_df = pd.DataFrame(high_corr_pairs, columns=["Feature 1", "Feature 2", "Correlation"])
display(high_corr_df)

## 4.3 PCA Benefits

In [ ]:
print("PCA Benefits:")
print("PCA reduces correlated features into fewer components.")
print("PCA reduces redundancy.")
print("PCA helps visualize high-dimensional data.")
print("PCA can reduce noise and improve clustering speed.")

# Part 5: Apply PCA

## 5.1 Fit PCA

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
print("PCA Completed")
print("PCA Shape:", X_pca_full.shape)

## 5.2 Eigenvalues

In [ ]:
eigenvalues = pca_full.explained_variance_
eigenvalue_df = pd.DataFrame({"Component": range(1, len(eigenvalues) + 1), "Eigenvalue": eigenvalues})
display(eigenvalue_df)

## 5.3 Explained Variance Ratio

In [ ]:
explained_variance = pca_full.explained_variance_ratio_
variance_df = pd.DataFrame({"Component": range(1, len(explained_variance) + 1), "Explained Variance Ratio": explained_variance})
display(variance_df)

## 5.4 Cumulative Variance

In [ ]:
cumulative_variance = np.cumsum(explained_variance)
cumulative_df = pd.DataFrame({"Component": range(1, len(cumulative_variance) + 1), "Cumulative Variance": cumulative_variance})
display(cumulative_df)

## 5.5 PCA Interpretation

In [ ]:
print("The first principal component captures the highest variance.")
print("Explained variance ratio shows how much information each component keeps.")
print("Cumulative variance helps decide how many components to keep.")

# Part 6: Choosing Components

## 6.1 Scree Plot

In [ ]:
components = range(1, len(explained_variance) + 1)
plt.figure(figsize=(7,5))
plt.plot(components, explained_variance, marker="o")
plt.title("Scree Plot")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.show()

## 6.2 Cumulative Explained Variance Plot

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(components, cumulative_variance, marker="o")
plt.axhline(y=0.90, linestyle="--")
plt.axhline(y=0.95, linestyle="--")
plt.axhline(y=0.99, linestyle="--")
plt.title("Cumulative Explained Variance Plot")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.show()

## 6.3 Components for 90%, 95%, and 99% Variance

In [ ]:
components_90 = np.argmax(cumulative_variance >= 0.90) + 1
components_95 = np.argmax(cumulative_variance >= 0.95) + 1
components_99 = np.argmax(cumulative_variance >= 0.99) + 1
print("Components needed for 90% variance:", components_90)
print("Components needed for 95% variance:", components_95)
print("Components needed for 99% variance:", components_99)

# Part 7: Visualization

## 7.1 2D PCA

In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
pca_2d_df = pd.DataFrame(X_pca_2d, columns=["PC1", "PC2"])
display(pca_2d_df.head())

## 7.2 Visualize 2D PCA

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(pca_2d_df["PC1"], pca_2d_df["PC2"], alpha=0.7)
plt.title("2D PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## 7.3 3D PCA

In [ ]:
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)
pca_3d_df = pd.DataFrame(X_pca_3d, columns=["PC1", "PC2", "PC3"])
display(pca_3d_df.head())

## 7.4 Visualize 3D PCA

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(pca_3d_df["PC1"], pca_3d_df["PC2"], pca_3d_df["PC3"], alpha=0.7)
ax.set_title("3D PCA Projection")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()

## 7.5 Visualization Interpretation

In [ ]:
print("2D and 3D PCA plots help visualize high-dimensional wine data.")
print("If points form groups, it may indicate natural clusters.")
print("If points overlap, wines may have similar chemical profiles.")

# Part 8: Feature Contribution

## 8.1 PCA Loadings

In [ ]:
loadings = pd.DataFrame(pca_2d.components_.T, columns=["PC1", "PC2"], index=X.columns)
display(loadings)

## 8.2 Most Influential Features for PC1

In [ ]:
pc1_features = loadings["PC1"].abs().sort_values(ascending=False)
display(pc1_features.head(5))

## 8.3 Most Influential Features for PC2

In [ ]:
pc2_features = loadings["PC2"].abs().sort_values(ascending=False)
display(pc2_features.head(5))

## 8.4 Least Influential Features

In [ ]:
overall_contribution = loadings.abs().sum(axis=1).sort_values()
display(overall_contribution.head(5))
print("Larger absolute loading means higher feature contribution.")

# Part 9: PCA + KMeans

## 9.1 KMeans Before PCA

In [ ]:
kmeans_original = KMeans(n_clusters=3, random_state=42, n_init=10)
start_time = time.time()
original_labels = kmeans_original.fit_predict(X_scaled)
original_time = time.time() - start_time
original_inertia = kmeans_original.inertia_
original_silhouette = silhouette_score(X_scaled, original_labels)
print("Inertia:", original_inertia)
print("Silhouette Score:", original_silhouette)
print("Training Time:", original_time)

## 9.2 KMeans After PCA

In [ ]:
pca_for_kmeans = PCA(n_components=components_95)
X_pca_kmeans = pca_for_kmeans.fit_transform(X_scaled)
kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
start_time = time.time()
pca_labels = kmeans_pca.fit_predict(X_pca_kmeans)
pca_time = time.time() - start_time
pca_inertia = kmeans_pca.inertia_
pca_silhouette = silhouette_score(X_pca_kmeans, pca_labels)
print("Inertia:", pca_inertia)
print("Silhouette Score:", pca_silhouette)
print("Training Time:", pca_time)

## 9.3 Compare KMeans Before and After PCA

In [ ]:
comparison_table = pd.DataFrame({
    "Model": ["KMeans Before PCA", "KMeans After PCA"],
    "Inertia": [original_inertia, pca_inertia],
    "Silhouette Score": [original_silhouette, pca_silhouette],
    "Training Time": [original_time, pca_time]
})
display(comparison_table)

## 9.4 Visualize PCA Clusters

In [ ]:
pca_2d_df["Cluster"] = pca_labels
plt.figure(figsize=(7,5))
plt.scatter(pca_2d_df["PC1"], pca_2d_df["PC2"], c=pca_2d_df["Cluster"], alpha=0.7)
plt.title("KMeans Clusters after PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()
print("PCA may reduce training time and remove redundant information before clustering.")

# Part 10: Theory

In [ ]:
print("PCA Theory Questions and Answers:")
print("1. PCA transforms original features into principal components.")
print("2. Scaling is required because PCA is affected by feature magnitude.")
print("3. Eigenvectors show principal component directions.")
print("4. Eigenvalues show how much variance each component captures.")
print("5. Explained variance shows how much information is retained.")
print("6. PCA can reduce noise by removing low-variance components.")
print("7. PCA is unsupervised because it does not use a target variable.")
print("8. PCA can make interpretation harder because components are combinations of original features.")
print("9. PCA is useful when features are correlated.")
print("10. PCA can improve clustering by reducing redundancy.")

# Final Conclusion

This notebook completed PCA analysis on the Wine dataset.

The workflow included data understanding, EDA, preprocessing, correlation analysis, PCA, component selection, visualization, feature contribution, PCA with KMeans, and theory answers.

PCA helped reduce dimensionality, preserve important variance, visualize wine data, and compare clustering before and after dimensionality reduction.
